In [2]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from numpy.ma.extras import unique
from polars import Null

from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [3]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations.")
display(artistDF.head(3))

There are 444888 unique artist entities after the removal of the disallowed occupations.


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q7161""","""Q36834""","""Q6581097""","""composer""","""male""","""Q29""",null,null,null,"""Q1413570""","""https://en.wikipedia.org/wiki/…","""533761c0-61cd-4d0e-b264-c71d5d…",null,null,"""Spain""","""sardana""","""José María Ventura Casas"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""


In [4]:
#inspect the different fields in the system that are error prone
#see the different genders in the dataframe
print(artistDF[["artist","genderLabel"]].unique().select(pl.col("genderLabel").value_counts(sort=True)).unnest("genderLabel"))
print(artistDF[["genderLabel"]].unique().select(pl.col("genderLabel")).to_series().to_list())

#see some genders who have unknown or none gender
noGender = artistDF.filter(
    pl.col("gender").is_null()
)
noGender = noGender.select(pl.col(["artist","gender","genderLabel"])).unique()
print(noGender)
#Save the file for individuals who do not have a gender
noGender.write_csv("./missingValuesStore/artistsEntitiesWithoutGenders.csv",)


shape: (30, 2)
┌──────────────────┬────────┐
│ genderLabel      ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ male             ┆ 331681 │
│ female           ┆ 112112 │
│ non-binary       ┆ 472    │
│ trans woman      ┆ 471    │
│ trans man        ┆ 107    │
│ …                ┆ …      │
│ non-binary woman ┆ 1      │
│ neutral sex      ┆ 1      │
│ demigirl         ┆ 1      │
│ gender agnostic  ┆ 1      │
│ demiboy          ┆ 1      │
└──────────────────┴────────┘
['intersex trans woman', 'demiboy', 'transmasculine', 'apagender', 'cisgender woman', 'androgyne', 'genderfluid', 'trans woman', 'takatāpui', 'neutral sex', 'demigirl', 'transfeminine', 'trans man', 'gender agnostic', 'intersex man', 'agender', 'undisclosed gender', 'intersex woman', 'non-binary woman', 'travesti', 'female', 'transgender', 'cisgender man', 'genderqueer', 'non-binary', None, 'two-spirit', 'intersex', 'male', 'bigender']
shape: (41, 3)
┌────────────┬──────

Checked the genders to ensure that error/unknown genders were successfully nulled.

In [5]:
#Null the artist labels for the individuals who do not have an artist label

noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artistLabel")).unique()
name_dict = {k: None for k in noNames["artistLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("artistLabel").replace(name_dict))

noNamesDF = artistDF.filter(
    pl.col("artistLabel").is_null(),
).select(pl.col(["artist","artistLabel"])).unique()
display(noNamesDF)
print(f"There are {len(noNamesDF)} artists with no default labels/labels in english.")
noNamesDF.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

artist,artistLabel
str,str
"""Q97230916""",null
"""Q139431861""",null
"""Q12555882""",null
"""Q4333575""",null
"""Q139440117""",null
…,…
"""Q20533911""",null
"""Q139483803""",null
"""Q133520515""",null


There are 14131 artists with no default labels/labels in english.


In [6]:
#The individuals with a country without an english label
#First let's see if there are any unknown or erroneous countries by using the Q-numbers and there are
artistDF.filter(
    pl.col('countryCitizenshipLabel').str.starts_with("Q")
).select(pl.col('countryCitizenshipLabel')).unique().to_series().to_list()

noLabel_Country = artistDF.filter(
    pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
).select(pl.col("countryCitizenshipLabel")).unique()
countryName_dict = {k: None for k in noLabel_Country["countryCitizenshipLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("countryCitizenshipLabel").replace(countryName_dict))
artistDF = artistDF.with_columns(
    countryCitizenship = pl.when(~pl.col('countryCitizenship').str.starts_with("Q")).then(None).otherwise(pl.col('countryCitizenship'))
)
noCountryLabel = artistDF.filter(
    pl.col("countryCitizenshipLabel").is_null(),
).select(pl.col(["countryCitizenship","countryCitizenshipLabel"])).unique()
display(noCountryLabel)
# noCountry = artistDF.filter(
#     pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
# ).select(pl.col("countryCitizenship")).unique()
# display(noCountry)
# print(f"There are {len(noCountry)} artists with a country with no labels. These will be cleaned out.")
# noCountry.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledCountries.csv")

countryCitizenship,countryCitizenshipLabel
str,str
"""Q2181059""",null
"""Q20581158""",null
null,null


In [7]:
#Genres with no labels in english/default language
noGenre = artistDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
).select(pl.col('genreLabel')).unique()
noGenreName_dict = {k: None for k in noGenre["genreLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("genreLabel").replace(noGenreName_dict))
artistDF = artistDF.with_columns(
    genre = pl.when(~pl.col('genre').str.starts_with("Q")).then(None).otherwise(pl.col('genre'))
)

noGenreLabel = artistDF.filter(
    pl.col("genreLabel").is_null(),
).select(pl.col(["artist","artistLabel","genre","genreLabel"])).unique()
display(noGenreLabel)
#print(f"There are {len(noGenre)} artists with a genre with no labels. These will be cleaned out.")
noGenreLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledGenresOrNoGenres.csv")

artist,artistLabel,genre,genreLabel
str,str,str,str
"""Q137438727""","""Christopher Ning""",null,null
"""Q6306947""","""Julian Antonisz""",null,null
"""Q5347588""","""Efraín Amador Piñero""",null,null
"""Q32186206""","""King Wu""",null,null
"""Q23901602""","""Eartheater""",null,null
…,…,…,…
"""Q100715225""","""Ricardo Rapoport""",null,null
"""Q3615982""","""Andrea Restori""",null,null
"""Q112080283""","""António de Santa Barbara""",null,null


In [8]:
#Record Labels without a label
noRecordLabelLabel = artistDF.filter(
    pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode())
).select(pl.col(["recordLabelLabel"])).unique()
recordLabel_dict = {k: None for k in noRecordLabelLabel["recordLabelLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("recordLabelLabel").replace(recordLabel_dict),
    recordLabel = pl.when(~pl.col('recordLabel').str.starts_with("Q")).then(None).otherwise(pl.col('recordLabel')),
)

recordLabelsWithNoLabel = artistDF.filter(
    pl.col("recordLabelLabel").is_null()
).select(["artist","artistLabel","recordLabel","recordLabelLabel"]).unique()
display(recordLabelsWithNoLabel)

recordLabelsWithNoLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledRecordLabelsOrNoRecordLabels.csv")

artist,artistLabel,recordLabel,recordLabelLabel
str,str,str,str
"""Q98381716""","""Caroline Wilkins""",null,null
"""Q1900862""","""Marko Zdralek""",null,null
"""Q123406957""","""Somechiyo Asakusa""",null,null
"""Q11772398""","""Małgorzata Walewska""",null,null
"""Q113383152""","""Léo Lanvin""",null,null
…,…,…,…
"""Q52616614""","""Reena Riot""",null,null
"""Q5868659""","""Fransheska""",null,null
"""Q4990740""","""Hedda Wrangel""",null,null


In [9]:
#Influences Labels without a label
noLabelsInfluencedBY = artistDF.filter(
    pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode())
).select(pl.col(["influencedBYLabel"])).unique()

influencedBName_dict = {k: None for k in noLabelsInfluencedBY["influencedBYLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("influencedBYLabel").replace(influencedBName_dict),
    influencedBY = pl.when(~pl.col('influencedBY').str.starts_with("Q")).then(None).otherwise(pl.col('influencedBY'))
)

entitiesWithNullInfluences = artistDF.filter(
    pl.col("influencedBYLabel").is_null()
).select(pl.col(["artist","artistLabel","influencedBY","influencedBYLabel"])).unique()
display(entitiesWithNullInfluences)
entitiesWithNullInfluences.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledInfluencesOrNoInfluences.csv")

artist,artistLabel,influencedBY,influencedBYLabel
str,str,str,str
"""Q1369895""","""Ester Textorius""",null,null
"""Q139188784""","""Chuck Carter""",null,null
"""Q95162816""","""Seiichirō Satō""",null,null
"""Q137655543""","""Tiziano Cavaliere""",null,null
"""Q2673547""","""Karel Veenendaal""",null,null
…,…,…,…
"""Q114046019""","""Ruby Nikara""",null,null
"""Q30942734""",null,null,null
"""Q13034746""","""Fredi Nest""",null,null


In [10]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

The size of the dataset now is 444888


In [11]:
#Look at selection of the musicBrainz table for any anomalies
mBtable = artistDF.select(pl.col(["artist","artistLabel","musicBrainzID"])).unique()
mB_issues = mBtable.filter(
    pl.col("musicBrainzID").str.contains("http")
).unique()
display(mB_issues)


mB_issues.write_csv("./missingValuesStore/artistsEntitiesWithFormattingIssuesInTheirMusicBrainzID.csv")
mB_users = mBtable.filter(
    pl.col("musicBrainzID").str.contains("user")
)
display(mB_users)


#Because it is two people who had these errored user links in there I will drop those two rows as their info will continue to be in the dataset
artistDF = artistDF.filter(
    (~pl.col("musicBrainzID").str.contains("user")) | (pl.col("musicBrainzID").is_null())
)

#Get the artist ID of the people with the links for theirs instead of the number
artistDF = artistDF.with_columns(
    musicBrainzID = pl.when(pl.col("musicBrainzID").str.contains("artist")).then(
        pl.col("musicBrainzID").str.split("/").list.last()
    ).otherwise(pl.col("musicBrainzID")),
)

#See a selection of artists to make sure changes were done correctly.
display(artistDF.filter((pl.col("artist") == "Q138387061") | (pl.col("artist") == "Q138413725") | (pl.col("artist") == "Q139919749")  ).unique())


artist,artistLabel,musicBrainzID
str,str,str
"""Q139623092""","""Mert Grey""","""https://musicbrainz.org/artist…"
"""Q138589085""","""Cyprian Błażejczyk""","""https://musicbrainz.org/artist…"
"""Q139919749""","""Biko Manus""","""https://musicbrainz.org/artist…"
"""Q140159484""","""Boyonset""","""https://musicbrainz.org/artist…"
"""Q12216002""","""Saadoun Jaber""","""https://musicbrainz.org/artist…"
"""Q140085977""","""Confidence Survival Eyong""","""https://musicbrainz.org/artist…"
"""Q138387061""","""Monster Fragment""","""https://musicbrainz.org/user/m…"
"""Q138413725""","""Konstantin Popov""","""https://musicbrainz.org/user/k…"


artist,artistLabel,musicBrainzID
str,str,str
"""Q138387061""","""Monster Fragment""","""https://musicbrainz.org/user/m…"
"""Q138413725""","""Konstantin Popov""","""https://musicbrainz.org/user/k…"


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q139919749""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q30""",null,null,null,null,null,"""0984057b-5ef5-4c92-80fa-585934…",null,null,"""United States""",null,"""Biko Manus"""
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q138387061""","""Q639669""","""Q6581097""","""musician""","""male""","""Q142""",null,null,null,"""Q223840""",null,"""f681a0d9-b9bb-4555-9b22-ada68f…",null,null,"""France""","""hardstyle""","""Monster Fragment"""


In [12]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

The size of the dataset now is 444888


In [13]:
#Look at the data in the statements tab to ensure they are all numbers
print(artistDF.schema)

#By checking the schema I can see that the statementCount is rightfully a series of numbers (Float64)
#Did a mean to confirm that there were no strings in there that could cause an issue
artistDF.select(pl.col("statementCount")).unique().mean()

Schema({'artist': String, 'occupation': String, 'gender': String, 'occupationLabel': String, 'genderLabel': String, 'countryCitizenship': String, 'influencedBY': String, 'influencedBYLabel': String, 'statementCount': Float64, 'genre': String, 'article': String, 'musicBrainzID': String, 'recordLabel': String, 'recordLabelLabel': String, 'countryCitizenshipLabel': String, 'genreLabel': String, 'artistLabel': String})


statementCount
f64
333.764925


In [14]:
#Look at the wiki column and see that every article has the wiki link in there by checking for those without
artistDF.filter(
    (~pl.col("article").str.contains("en.wikipedia"))
).unique()

#From this I can see that all the articles in here are correct links by checking that they contain en.wikipedia

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str


In [15]:
artistDF = artistDF.unique()

At this point I have checked all the fields and attempted to clean up the data to highlight any unknowns. Each row should now be unique and contain a single value

In [16]:
#Look at the gender split of the data
#Discover the gender gap in the set and the number of people who have a null/unknown gender
artistsByGender = artistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
).unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGrouped.write_csv("./Visuals Data/artistsGroupedByGender.csv", null_value="null/unknown")
artistGenderGroupedFRAC = artistsByGender["genderLabel"].value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q130477254""","""non-binary woman""",1
"""Q124850662""","""apagender""",1
"""Q138806923""","""intersex trans woman""",5
"""Q124637723""","""gender agnostic""",1
"""Q48270""","""non-binary""",472
…,…,…
"""Q505371""","""agender""",12
"""Q93954933""","""demiboy""",1
"""Q52261234""","""neutral sex""",1


genderLabel,fraction
str,f64
"""male""",0.745138
"""female""",0.251865
"""non-binary""",0.00106
"""trans woman""",0.001058
"""trans man""",0.00024
…,…
"""non-binary woman""",0.000002
"""gender agnostic""",0.000002
"""apagender""",0.000002


In [17]:
#Look at the region/country split of the data
regions = artistDF.select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
).unique()
regionsGrouped = regions.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count())
regionsGrouped.write_csv("./Visuals Data/artistsGroupedByRegions-Countries.csv", null_value="null/unknown")
display(regionsGrouped.sample(5))

countRegionsNull = artistDF.filter(
    pl.col("countryCitizenship").is_null()
).select(pl.col("artist")).unique()

countRegionsNotNull = artistDF.filter(
    pl.col("countryCitizenship").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countRegionsNull)} artists without a region/country")
print(f"There are {len(countRegionsNotNull)} artists with a region/country")



countryCitizenship,countryCitizenshipLabel,artist
str,str,u32
"""Q1412""","""Finnish""",1
"""Q813""","""Kyrgyzstan""",79
"""Q12060881""","""Chinese Empire""",1
"""Q212""","""Ukraine""",2543
"""Q858""","""Syria""",187


There are 122946 artists without a region/country
There are 321943 artists with a region/country


In [18]:
#Look at the genre and gender-genre split on wikidata
genresDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("genre"),
    pl.col("genreLabel"),
).unique()
genresDFGrouped = genresDF.group_by(["genre", "genreLabel"]).agg(pl.col("artist").count())
genresDFGrouped.write_csv("./Visuals Data/artistsGroupedByGenres.csv", null_value="null/unknown")
display(genresDFGrouped.sample(5))
genreGenderSplit = genresDF.group_by(["genre", "genreLabel","genderLabel"]).agg(pl.col("artist").count())
print(genreGenderSplit.head())
genreGenderPivot = genreGenderSplit.pivot(
    "genderLabel",
    index = ["genre", "genreLabel"],
    values = "artist",
    aggregate_function = "sum"
)
genreGenderPivot.write_csv("./Visuals Data/artistsGroupedByGenresAndGender.csv", null_value="null/unknown")


countGenreNull = artistDF.filter(
    pl.col("genre").is_null()
).select(pl.col("artist")).unique()

countGenreNotNull = artistDF.filter(
    pl.col("genre").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countGenreNull)} artists without a genre")
print(f"There are {len(countGenreNotNull)} artists with a genre")

genreGenderPivot


genre,genreLabel,artist
str,str,u32
"""Q124357128""","""gabberpop""",4
"""Q11911997""","""canto valenciano""",1
"""Q16148439""","""Gaelic music""",1
"""Q5515214""","""gaana""",2
"""Q3191747""","""cadence rampa""",6


shape: (5, 4)
┌───────────┬────────────────────────────┬─────────────┬────────┐
│ genre     ┆ genreLabel                 ┆ genderLabel ┆ artist │
│ ---       ┆ ---                        ┆ ---         ┆ ---    │
│ str       ┆ str                        ┆ str         ┆ u32    │
╞═══════════╪════════════════════════════╪═════════════╪════════╡
│ Q7832297  ┆ traditional Japanese music ┆ female      ┆ 1      │
│ Q16323732 ┆ shaabi                     ┆ male        ┆ 3      │
│ Q11787389 ┆ club/dance music           ┆ male        ┆ 9      │
│ Q722989   ┆ Hungarian folk music       ┆ male        ┆ 1      │
│ Q497957   ┆ nueva trova                ┆ female      ┆ 1      │
└───────────┴────────────────────────────┴─────────────┴────────┘
There are 345319 artists without a genre
There are 99569 artists with a genre


genre,genreLabel,female,male,cisgender woman,genderqueer,transfeminine,non-binary,genderfluid,non-binary woman,trans man,trans woman,cisgender man,travesti,null,agender,transgender,intersex woman,two-spirit,intersex trans woman,undisclosed gender,transmasculine,intersex,takatāpui,demigirl,demiboy,bigender,neutral sex,intersex man,apagender,androgyne,gender agnostic
str,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""Q7832297""","""traditional Japanese music""",1,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q16323732""","""shaabi""",0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q11787389""","""club/dance music""",9,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q722989""","""Hungarian folk music""",1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q497957""","""nueva trova""",1,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q14493900""","""Texas country music""",0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q64137792""","""religious vocal music""",0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q12307741""","""dansktop""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [19]:
#Look at the occupation and occupation/gender split
occupationDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("occupation"),
    pl.col("occupationLabel"),
).unique()
occupationDFDFGrouped = occupationDF.group_by(["occupation", "occupationLabel"]).agg(pl.col("artist").count())
occupationDFDFGrouped.write_csv("./Visuals Data/artistsGroupedByOccupation.csv", null_value="null/unknown")
display(occupationDFDFGrouped.sample(5))
occupationGenderSplit = occupationDF.group_by(["occupation", "occupationLabel","genderLabel"]).agg(pl.col("artist").count())

occupationGenderPivot = occupationGenderSplit.pivot(
    "genderLabel",
    index = ["occupation", "occupationLabel"],
    values = "artist",
    aggregate_function = "sum"
)

occupationGenderPivot.write_csv("./Visuals Data/artistsGroupedByOccupationAndGender.csv", null_value="null/unknown")
display(occupationGenderPivot.sample(5))

occupation,occupationLabel,artist
str,str,u32
"""Q99510107""","""sufi singer""",9
"""Q502323""","""goze""",2
"""Q4386521""","""biwa hÅshi""",1
"""Q1792566""","""whistle artist""",21
"""Q1294626""","""music artist""",342


occupation,occupationLabel,male,non-binary,trans woman,female,null,undisclosed gender,agender,intersex man,transfeminine,cisgender woman,intersex trans woman,genderqueer,two-spirit,genderfluid,transmasculine,trans man,takatāpui,demiboy,intersex woman,transgender,travesti,apagender,neutral sex,bigender,intersex,cisgender man,androgyne,demigirl,non-binary woman,gender agnostic
str,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""Q84488218""","""heavy metal bass guitarist""",3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q3074722""","""recorder player""",82,0,0,53,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q113621223""","""soukous musician""",2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q67200504""","""heavy metal singer""",15,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q61996187""","""classical pianist""",377,0,0,266,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Old Code that I will ignore for now

In [20]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode()),
    ~pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode()),
    ~pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode()),
    ~pl.col("genre").is_in(pl.col("genreLabel").implode()),
)

#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.head(5))
display(cleanedArtistDF.shape)

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel,isThereAWiki
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,bool
"""Q224650""","""Q486748""","""Q6581072""","""pianist""","""female""","""Q16""","""Q234691""","""Stevie Nicks""",222.0,"""Q378988""","""https://en.wikipedia.org/wiki/…","""ce58d854-7430-4231-aa44-97f014…","""Q1029135""","""Nettwerk""","""Canada""","""adult contemporary music""","""Sarah McLachlan""",true
"""Q180453""","""Q36834""","""Q6581072""","""composer""","""female""","""Q30""","""Q273080""","""Patsy Cline""",506.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""1d543e07-d0d2-4834-a8db-d65c50…","""Q2386070""","""Sugar Hill Records""","""United States""","""pop music""","""Dolly Parton""",true
"""Q17403494""","""Q183945""","""Q6581072""","""record producer""","""female""","""Q30""","""Q303""","""Elvis Presley""",null,"""Q131272""","""https://en.wikipedia.org/wiki/…","""b2029169-2574-4305-820f-252a5f…","""Q202585""","""RCA Records""","""United States""","""soul""","""Meghan Trainor""",true
"""Q193645""","""Q753110""","""Q6581097""","""songwriter""","""male""","""Q30""","""Q4030""","""Duke Ellington""",410.0,"""Q164444""","""https://en.wikipedia.org/wiki/…","""5803c81e-739a-4057-9a5c-cf84e5…","""Q287177""","""ABC Records""","""United States""","""funk""","""Quincy Jones""",true
"""Q180453""","""Q753110""","""Q6581072""","""songwriter""","""female""","""Q30""","""Q272931""","""Loretta Lynn""",506.0,"""Q180268""","""https://en.wikipedia.org/wiki/…","""1d543e07-d0d2-4834-a8db-d65c50…","""Q165745""","""Mercury Records""","""United States""","""gospel music""","""Dolly Parton""",true


(137036, 18)

That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [21]:
#Discover the gender gap in the set.
artistsByGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistsByGender = artistsByGender.unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistsByGender["genderLabel"].value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q6581072""","""female""",188
"""Q1052281""","""trans woman""",1
"""Q48270""","""non-binary""",5
"""Q27679684""","""transfeminine""",1
"""Q6581097""","""male""",305
"""Q18116794""","""genderfluid""",4


genderLabel,fraction
str,f64
"""male""",0.605159
"""female""",0.373016
"""non-binary""",0.009921
"""genderfluid""",0.007937
"""trans woman""",0.001984
"""transfeminine""",0.001984


In [22]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

alt.Chart(...)

alt.LayerChart(...)

In [23]:
#Top genres and genre gap

In [24]:
#Top occupations

In [25]:
#Top occupations by gender

In [26]:
#Gender and genre split